In [2]:
# -------------------------------
# 1. IMPORT LIBRARIES
# -------------------------------

import numpy as np
import pandas as pd
import yfinance as yf
import datetime as dt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error

import statsmodels.api as sm
from statsmodels.tsa.stattools import adfuller


# -------------------------------
# 2. DOWNLOAD DATA
# -------------------------------

end = dt.date.today()
start = end - dt.timedelta(days=365*5)

tickers = ['HDFCBANK.NS','^NSEI','^INDIAVIX','^NSEBANK']

df = yf.download(tickers=tickers, start=start, end=end, auto_adjust=True)['Close']


# -------------------------------
# 3. FEATURE ENGINEERING
# -------------------------------

# Log returns (better for modeling)
df['hdfc_returns'] = np.log(df['HDFCBANK.NS'] / df['HDFCBANK.NS'].shift(1))
df['nse_returns'] = np.log(df['^NSEI'] / df['^NSEI'].shift(1))
df['bank_returns'] = np.log(df['^NSEBANK'] / df['^NSEBANK'].shift(1))

# Lagged returns (previous info only)
df['hdfc_lagged'] = df['hdfc_returns'].shift(1)

# ⚠️ FIX: avoid data leakage → use shift(1)
df['volatility'] = df['hdfc_returns'].rolling(20).std().shift(1)

# ⚠️ FIX: momentum should use past info
df['momentum'] = df['HDFCBANK.NS'].pct_change(5).shift(1)

# ⚠️ FIX: use VIX returns instead of raw level
df['vix'] = np.log(df['^INDIAVIX'] / df['^INDIAVIX'].shift(1))


# -------------------------------
# 4. CLEAN DATA
# -------------------------------

# Drop original price columns
df.drop(columns=['HDFCBANK.NS','^NSEI','^NSEBANK','^INDIAVIX'], inplace=True)

# Drop NaNs from shifting
df.dropna(inplace=True)

# Lowercase column names
df.columns = df.columns.str.lower()


# -------------------------------
# 5. ADF TEST (STATIONARITY)
# -------------------------------

def adf_test(series, name=""):
    result = adfuller(series.dropna())

    print(f"\nADF Test → {name}")
    print(f"ADF Statistic: {result[0]:.2f}")
    print(f"p-value: {result[1]:.4f}")

    if result[1] < 0.05:
        print("→ Stationary ✅")
    else:
        print("→ Non-stationary ❌")

# Run for all columns
for col in df.columns:
    adf_test(df[col], col)


# -------------------------------
# 6. TRAIN / TEST SPLIT
# -------------------------------

train_size = int(len(df) * 0.8)

train_df = df.iloc[:train_size]
test_df = df.iloc[train_size:]


# -------------------------------
# 7. DEFINE FEATURES (CONSISTENT)
# -------------------------------

features = ['hdfc_lagged','bank_returns','momentum']

X_train = train_df[features]
y_train = train_df['hdfc_returns']

X_test = test_df[features]
y_test = test_df['hdfc_returns']


# -------------------------------
# 8. TRAIN MODEL (SKLEARN)
# -------------------------------

model = LinearRegression()
model.fit(X_train, y_train)


# -------------------------------
# 9. TRAIN MODEL (STATSMODELS)
# -------------------------------

X_train_sm = sm.add_constant(X_train)
model_sm = sm.OLS(y_train, X_train_sm).fit()

# -------------------------------
# 10. MODEL DIAGNOSTICS
# -------------------------------

print("\n===== MODEL STATS =====")

print("R²:", model_sm.rsquared)
print("Adj R²:", model_sm.rsquared_adj)
print("F-test p-value:", model_sm.f_pvalue)

print("\nCoefficients:")
print(model_sm.params)

print("\np-values:")
print(model_sm.pvalues)


# -------------------------------
# 11. TRAIN PERFORMANCE
# -------------------------------

y_pred_train = model.predict(X_train)

rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
mae_train = mean_absolute_error(y_train, y_pred_train)

print("\nTrain RMSE:", rmse_train)
print("Train MAE:", mae_train)


# -------------------------------
# 12. TEST PERFORMANCE
# -------------------------------

y_pred_test = model.predict(X_test)

rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_test = mean_absolute_error(y_test, y_pred_test)

print("\nTest RMSE:", rmse_test)
print("Test MAE:", mae_test)


# -------------------------------
# 13. OVERFITTING CHECK
# -------------------------------

if rmse_test > rmse_train:
    print("\n⚠️ Model may be overfitting")
else:
    print("\n✅ Model generalizes well")

[*********************100%***********************]  4 of 4 completed



ADF Test → hdfc_returns
ADF Statistic: -33.71
p-value: 0.0000
→ Stationary ✅

ADF Test → nse_returns
ADF Statistic: -34.99
p-value: 0.0000
→ Stationary ✅

ADF Test → bank_returns
ADF Statistic: -15.70
p-value: 0.0000
→ Stationary ✅

ADF Test → hdfc_lagged
ADF Statistic: -33.88
p-value: 0.0000
→ Stationary ✅

ADF Test → volatility
ADF Statistic: -2.86
p-value: 0.0496
→ Stationary ✅

ADF Test → momentum
ADF Statistic: -7.18
p-value: 0.0000
→ Stationary ✅

ADF Test → vix
ADF Statistic: -8.35
p-value: 0.0000
→ Stationary ✅

===== MODEL STATS =====
R²: 0.5920558540016801
Adj R²: 0.5907689639512123
F-test p-value: 1.3137205471488638e-184

Coefficients:
const          -0.000133
hdfc_lagged     0.033706
bank_returns    0.936971
momentum       -0.018094
dtype: float64

p-values:
const            6.314227e-01
hdfc_lagged      1.488052e-01
bank_returns    4.275593e-187
momentum         7.546271e-02
dtype: float64

Train RMSE: 0.008507712810670216
Train MAE: 0.006137714531890825

Test RMSE: 0.006